In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".csv"):
            print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thư mục chứa dữ liệu
data_path = "/kaggle/input/competitions/shai-music-genre-classification/"

# Đọc ba file
train = pd.read_csv(data_path + "train (1).csv")
test = pd.read_csv(data_path + "test (2).csv")
sample_submission = pd.read_csv(data_path + "sample_submission (2).csv")

# Kiểm tra kích thước
print("Train:", train.shape)
print("Test:", test.shape)
print("Submission:", sample_submission.shape)

# Xem 5 dòng đầu tiên
train.head()

In [ ]:
train.info()

#### Thống kê dữ liệu thiếu

In [ ]:
missing = pd.DataFrame({
    "So luong thieu": train.isna().sum(),
    "Ty le thieu (%)": train.isna().mean() * 100
})

missing = missing[missing["So luong thieu"] > 0]
missing.sort_values("So luong thieu", ascending=False).round(2)

In [ ]:
class_summary = pd.DataFrame({
    "So luong": train["Class"].value_counts().sort_index(),
    "Ty le (%)": train["Class"].value_counts(
        normalize=True
    ).sort_index() * 100
})

print(class_summary.round(2))

plt.figure(figsize=(10, 4))
sns.countplot(data=train, x="Class", color="steelblue")
plt.title("Phan bo cac lop nhac")
plt.xlabel("Class")
plt.ylabel("So luong bai hat")
plt.tight_layout()
plt.show()

In [ ]:
print("So dong trung hoan toan:", train.duplicated().sum())

print(
    "So dong lap theo nghe si va ten bai:",
    train.duplicated(
        subset=["Artist Name", "Track Name"]
    ).sum()
)

In [ ]:
print("DU LIEU THIEU:")
print(train.isna().sum()[train.isna().sum() > 0])

print("\nTRUNG TOAN BO DONG:")
print(train.duplicated().sum())

print("\nTRUNG NGHE SI VA TEN BAI:")
print(train.duplicated(subset=["Artist Name", "Track Name"]).sum())

In [ ]:
song_cols = ["Artist Name", "Track Name"]

# keep=False: lấy tất cả các dòng thuộc nhóm bài trùng
duplicate_songs = train[
    train.duplicated(subset=song_cols, keep=False)
]

print("Tong so dong thuoc cac nhom trung:", len(duplicate_songs))

duplicate_songs.sort_values(song_cols).head(20)

In [ ]:
# Đếm số nhãn khác nhau của mỗi bài
labels_per_song = train.groupby(song_cols)["Class"].nunique()

conflicting_songs = labels_per_song[labels_per_song > 1]

print("So nhom bai hat co nhieu nhan:", len(conflicting_songs))
print(conflicting_songs.head(10))

In [ ]:
train.drop(columns=["Id", "Class"]).describe().T

In [ ]:
# Chuẩn hóa tên để nhận diện cùng bài hát
song_info = train[["Artist Name", "Track Name"]].fillna("")

song_info = song_info.apply(
    lambda col: col.str.strip().str.lower()
)

# Mỗi cặp nghệ sĩ + tên bài có một mã nhóm
song_groups = song_info.groupby(
    ["Artist Name", "Track Name"]
).ngroup()

print("So nhom bai hat:", song_groups.nunique())

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# V1 dùng các đặc trưng số
X = train.drop(columns=["Id", "Artist Name", "Track Name", "Class"])
y = train["Class"]

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Lấy một fold làm validation
train_idx, valid_idx = next(
    splitter.split(X, y, groups=song_groups)
)

X_train = X.iloc[train_idx].copy()
X_valid = X.iloc[valid_idx].copy()

y_train = y.iloc[train_idx].copy()
y_valid = y.iloc[valid_idx].copy()

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

In [ ]:
train_groups = set(song_groups.iloc[train_idx])
valid_groups = set(song_groups.iloc[valid_idx])

print("So nhom trung giua hai tap:",
      len(train_groups.intersection(valid_groups)))

class_ratio = pd.DataFrame({
    "Train (%)": y_train.value_counts(normalize=True) * 100,
    "Valid (%)": y_valid.value_counts(normalize=True) * 100
}).sort_index()

print(class_ratio.round(2))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

model_v1 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Huấn luyện trên tập train đã chia
model_v1.fit(X_train, y_train)

# Giữ đúng các cột và thứ tự đặc trưng
X_test = test[X_train.columns]

test_predictions = model_v1.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "Class": test_predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.shape)
submission.head()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

pred_train_v1 = model_v1.predict(X_train)
pred_valid_v1 = model_v1.predict(X_valid)

print("V1 Train Accuracy:",
      accuracy_score(y_train, pred_train_v1))
print("V1 Valid Accuracy:",
      accuracy_score(y_valid, pred_valid_v1))
print("V1 Valid Macro F1:",
      f1_score(y_valid, pred_valid_v1, average="macro"))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier

model_v2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", ExtraTreesClassifier(
        n_estimators=400,
        random_state=42,
        n_jobs=-1
    ))
])

model_v2.fit(X_train, y_train)

In [ ]:
pred_train_v2 = model_v2.predict(X_train)
pred_valid_v2 = model_v2.predict(X_valid)

comparison = pd.DataFrame({
    "Model": ["V1 - Random Forest", "V2 - Extra Trees"],
    "Train Accuracy": [
        accuracy_score(y_train, pred_train_v1),
        accuracy_score(y_train, pred_train_v2)
    ],
    "Valid Accuracy": [
        accuracy_score(y_valid, pred_valid_v1),
        accuracy_score(y_valid, pred_valid_v2)
    ],
    "Valid Macro F1": [
        f1_score(y_valid, pred_valid_v1, average="macro"),
        f1_score(y_valid, pred_valid_v2, average="macro")
    ]
})

print(comparison.round(4).to_string(index=False))

In [ ]:
print(comparison.round(4).to_string(index=False))

In [ ]:
from sklearn.metrics import classification_report

# Kết quả từng lớp của mô hình tốt nhất hiện tại
print("BAO CAO V1:")
print(classification_report(
    y_valid,
    pred_valid_v1,
    digits=4,
    zero_division=0
))

# Kiểm tra các dòng có cùng đặc trưng số nhưng khác nhãn
feature_cols = X.columns.tolist()

labels_per_features = train.groupby(
    feature_cols,
    dropna=False
)["Class"].nunique()

print(
    "So nhom dac trung giong nhau nhung khac nhan:",
    (labels_per_features > 1).sum()
)

In [ ]:
from sklearn.base import clone

# Tạo mô hình riêng để huấn luyện trên toàn bộ dữ liệu
# Giữ model_v1 ban đầu để bảo toàn kết quả validation
final_model = clone(model_v1)

X_full = train[X_train.columns]
y_full = train["Class"]

final_model.fit(X_full, y_full)

# Dự đoán tập test
test_predictions = final_model.predict(test[X_train.columns])

# Tạo file đúng định dạng
submission = pd.DataFrame({
    "Id": test["Id"],
    "Class": test_predictions.astype(int)
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Da tao submission.csv")
print("Kich thuoc:", submission.shape)
print(submission.head())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

model_v3 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

model_v3.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

pred_train_v3 = model_v3.predict(X_train)
pred_valid_v3 = model_v3.predict(X_valid)

result_v3 = pd.DataFrame({
    "Model": ["V3 - Random Forest regularized"],
    "Train Accuracy": [
        accuracy_score(y_train, pred_train_v3)
    ],
    "Valid Accuracy": [
        accuracy_score(y_valid, pred_valid_v3)
    ],
    "Valid Macro F1": [
        f1_score(y_valid, pred_valid_v3, average="macro")
    ]
})

comparison_v3 = pd.concat(
    [comparison, result_v3],
    ignore_index=True
)

print(comparison_v3.round(4).to_string(index=False))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_valid,
    pred_valid_v3,
    digits=4,
    zero_division=0
))

#### V4

In [ ]:
from sklearn.base import clone

# Sao chép cấu hình V1, không thay đổi mô hình V1
model_v4 = clone(model_v1)

# Tăng trọng số cho các lớp ít mẫu
model_v4.set_params(model__class_weight="balanced")

model_v4.fit(X_train, y_train)

In [ ]:
pred_train_v4 = model_v4.predict(X_train)
pred_valid_v4 = model_v4.predict(X_valid)

result_v4 = pd.DataFrame({
    "Model": ["V4 - Random Forest balanced"],
    "Train Accuracy": [
        accuracy_score(y_train, pred_train_v4)
    ],
    "Valid Accuracy": [
        accuracy_score(y_valid, pred_valid_v4)
    ],
    "Valid Macro F1": [
        f1_score(y_valid, pred_valid_v4, average="macro")
    ]
})

comparison_v4 = pd.concat(
    [comparison_v3, result_v4],
    ignore_index=True
)

print(comparison_v4.round(4).to_string(index=False))

In [ ]:
print(classification_report(
    y_valid,
    pred_valid_v4,
    digits=4,
    zero_division=0
))

#### V5

In [ ]:
# Giữ các đặc trưng số cũ và thêm tên nghệ sĩ
numeric_cols = X_train.columns.tolist()
feature_cols_v5 = numeric_cols + ["Artist Name"]

X_train_v5 = train.iloc[train_idx][feature_cols_v5].copy()
X_valid_v5 = train.iloc[valid_idx][feature_cols_v5].copy()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor_v5 = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric_cols),

    ("artist", Pipeline([
        ("imputer", SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), ["Artist Name"])
])

model_v5 = Pipeline([
    ("preprocessor", preprocessor_v5),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

model_v5.fit(X_train_v5, y_train)

In [ ]:
pred_train_v5 = model_v5.predict(X_train_v5)
pred_valid_v5 = model_v5.predict(X_valid_v5)

comparison_v5 = pd.DataFrame({
    "Model": ["V1 - Numeric", "V5 - Numeric + Artist"],
    "Train Accuracy": [
        accuracy_score(y_train, pred_train_v1),
        accuracy_score(y_train, pred_train_v5)
    ],
    "Valid Accuracy": [
        accuracy_score(y_valid, pred_valid_v1),
        accuracy_score(y_valid, pred_valid_v5)
    ],
    "Valid Macro F1": [
        f1_score(y_valid, pred_valid_v1, average="macro"),
        f1_score(y_valid, pred_valid_v5, average="macro")
    ]
})

print(comparison_v5.round(4).to_string(index=False))

print(classification_report(
    y_valid,
    pred_valid_v5,
    digits=4,
    zero_division=0
))

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

X_numeric = train[numeric_cols]
X_artist = train[feature_cols_v5]
y_all = train["Class"]

cv_results = []

for fold, (tr_idx, va_idx) in enumerate(
    cv.split(X_numeric, y_all, groups=song_groups),
    start=1
):
    models = [
        ("V1", model_v1, X_numeric),
        ("V5", model_v5, X_artist)
    ]

    for name, base_model, features in models:
        model = clone(base_model)

        model.fit(
            features.iloc[tr_idx],
            y_all.iloc[tr_idx]
        )

        predictions = model.predict(features.iloc[va_idx])

        cv_results.append({
            "Fold": fold,
            "Model": name,
            "Accuracy": accuracy_score(
                y_all.iloc[va_idx], predictions
            ),
            "Macro F1": f1_score(
                y_all.iloc[va_idx],
                predictions,
                average="macro"
            )
        })

    print(f"Da xong fold {fold}/5")

cv_results = pd.DataFrame(cv_results)

print("\nKET QUA TUNG FOLD:")
print(cv_results.round(4).to_string(index=False))

print("\nTRUNG BINH VA DO LECH CHUAN:")
print(
    cv_results.groupby("Model")[["Accuracy", "Macro F1"]]
    .agg(["mean", "std"])
    .round(4)
)

In [ ]:
from sklearn.base import clone

# Huấn luyện V5 trên toàn bộ dữ liệu có nhãn
final_model_v5 = clone(model_v5)

final_model_v5.fit(
    train[feature_cols_v5],
    train["Class"]
)

# Dự đoán tập test
test_predictions_v5 = final_model_v5.predict(
    test[feature_cols_v5]
)

# Xuất file riêng để phân biệt với submission V1
submission_v5 = pd.DataFrame({
    "Id": test["Id"],
    "Class": test_predictions_v5.astype(int)
})

submission_v5.to_csv(
    "/kaggle/working/submission_v5.csv",
    index=False
)

print("Da tao submission_v5.csv")
print("Kich thuoc:", submission_v5.shape)
print(submission_v5.head())

#### v6

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

X_v6 = train[feature_cols_v5].copy()

# CatBoost cần cột phân loại không có giá trị thiếu
X_v6["Artist Name"] = X_v6["Artist Name"].fillna("Unknown")

X_train_v6 = X_v6.iloc[train_idx].copy()
X_valid_v6 = X_v6.iloc[valid_idx].copy()

In [ ]:
model_v6 = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

model_v6.fit(
    X_train_v6,
    y_train,
    cat_features=["Artist Name"],
    eval_set=(X_valid_v6, y_valid),
    early_stopping_rounds=100,
    use_best_model=True
)

In [ ]:
pred_train_v6 = model_v6.predict(X_train_v6).ravel().astype(int)
pred_valid_v6 = model_v6.predict(X_valid_v6).ravel().astype(int)

comparison_v6 = pd.DataFrame({
    "Model": ["V5 - Random Forest + Artist", "V6 - CatBoost + Artist"],
    "Train Accuracy": [
        accuracy_score(y_train, pred_train_v5),
        accuracy_score(y_train, pred_train_v6)
    ],
    "Valid Accuracy": [
        accuracy_score(y_valid, pred_valid_v5),
        accuracy_score(y_valid, pred_valid_v6)
    ],
    "Valid Macro F1": [
        f1_score(y_valid, pred_valid_v5, average="macro"),
        f1_score(y_valid, pred_valid_v6, average="macro")
    ]
})

print(comparison_v6.round(4).to_string(index=False))
print("So cay duoc giu:", model_v6.tree_count_)

print(classification_report(
    y_valid,
    pred_valid_v6,
    digits=4,
    zero_division=0
))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

cv_v6 = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results_v6 = []

for fold, (tr_idx, va_idx) in enumerate(
    cv_v6.split(X_v6, train["Class"], groups=song_groups),
    start=1
):
    fold_model = CatBoostClassifier(
        iterations=774,
        learning_rate=0.05,
        depth=6,
        loss_function="MultiClass",
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )

    fold_model.fit(
        X_v6.iloc[tr_idx],
        train["Class"].iloc[tr_idx],
        cat_features=["Artist Name"]
    )

    predictions = (
        fold_model.predict(X_v6.iloc[va_idx])
        .ravel()
        .astype(int)
    )

    results_v6.append({
        "Fold": fold,
        "Accuracy": accuracy_score(
            train["Class"].iloc[va_idx], predictions
        ),
        "Macro F1": f1_score(
            train["Class"].iloc[va_idx],
            predictions,
            average="macro"
        )
    })

    print(f"Da xong fold {fold}/5")

results_v6 = pd.DataFrame(results_v6)

print("\nKET QUA TUNG FOLD:")
print(results_v6.round(4).to_string(index=False))

print("\nTRUNG BINH VA DO LECH CHUAN:")
print(results_v6[["Accuracy", "Macro F1"]].agg(["mean", "std"]).round(4))

In [ ]:
# Huấn luyện bản cuối trên toàn bộ dữ liệu có nhãn
final_model_v6 = CatBoostClassifier(
    iterations=774,
    learning_rate=0.05,
    depth=6,
    loss_function="MultiClass",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

final_model_v6.fit(
    X_v6,
    train["Class"],
    cat_features=["Artist Name"]
)

# Chuẩn bị test giống lúc huấn luyện
X_test_v6 = test[feature_cols_v5].copy()
X_test_v6["Artist Name"] = (
    X_test_v6["Artist Name"].fillna("Unknown")
)

# Dự đoán nhãn
test_predictions_v6 = (
    final_model_v6.predict(X_test_v6)
    .ravel()
    .astype(int)
)

# Tạo file nộp bài
submission_v6 = pd.DataFrame({
    "Id": test["Id"],
    "Class": test_predictions_v6
})

submission_v6.to_csv(
    "/kaggle/working/submission_v6.csv",
    index=False
)

print("Da tao submission_v6.csv")
print("Kich thuoc:", submission_v6.shape)
print(submission_v6.head())